# Protenix Pilot10 — native-mapped US-align evaluation (CPU, Run All)

**Same frozen ten RNAs and native-reference bundle as previous six models.** No GPU, no re-inference, no manual upload. Converts the **validated finite-coordinate** Protenix v2 CIF output to a numbered PDB, trims only to experimentally observed RNA residues, then computes native-normalized TM-score and RMSD with pinned US-align v20260527.

**Caveats:** exploratory n=10; Protenix uses RNA-only no MSA/no-template mode, while previous models have different input conditions; some experimental chains have missing residues; training-set overlap/template scarcity not cleared. Existing inference results remain unchanged.

## 1 — Mount Google Drive and verify input roots

In [ ]:
from pathlib import Path
from google.colab import drive
import os, csv, json, hashlib, re, math, subprocess, shutil, zipfile, statistics
from datetime import datetime, timezone
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/RNA3D_Benchmark')
PRED_ROOT=ROOT/'Protenix/runs/protenix_pilot10_rna_only_nomsa_notemplate_v2_layernormtorch'
NATIVE_ZIP=ROOT/'DRfold/Evaluations/RNA3D_DRfold_Pilot10_Eval_Inputs_20261119.zip'
EVAL_ROOT=ROOT/'Protenix/Evaluations'
RUN_NAME='protenix_pilot10_layernormtorch_usalign20260527_20261120_v1'
TARGETS=['8ubd_H','9fia_bJ','9cfn_B','9m31_B','9g4p_B','9lq7_K','8z8u_B','9g6k_lB','9cf2_W','8wf8_B']
MODEL='protenix_base_default_v1.0.0'
UPSTREAM_SHA='85767b811c40ed46e73a9b39519cf6bfca8701ba'
USALIGN_SHA='177cc8a2bbd3e2a6e9c5faaaa4ff5dfa1e6048f7'
assert PRED_ROOT.is_dir(),f'Missing Protenix v2 predictions: {PRED_ROOT}'
assert NATIVE_ZIP.is_file(),f'Missing frozen native references: {NATIVE_ZIP}'
assert NATIVE_ZIP.stat().st_size==395773,'Reference archive size has unexpectedly changed'
with zipfile.ZipFile(NATIVE_ZIP) as z:
    assert z.testzip() is None
    for t in TARGETS:assert f'native/{t}_native_eval.pdb' in z.namelist(),t
for t in TARGETS:
    assert (PRED_ROOT/t/'prediction.cif').is_file(),f'{t}: CIF absent'
    assert (PRED_ROOT/t/'complete.json').is_file(),f'{t}: completion marker absent'
print('✅ Preflight: 10 CIFs + 10 completion markers + 10 frozen native references')

## 2 — Install CIF parser and build pinned US-align (CPU)

In [ ]:
%pip -q install gemmi==0.7.3
import gemmi
SRC=Path('/content/USalign_protenix_pilot10_20260527')
if not SRC.is_dir():
    subprocess.run(['git','clone','--depth','1','--branch','20260527','https://github.com/pylelab/USalign.git',str(SRC)],check=True)
actual=subprocess.check_output(['git','-C',str(SRC),'rev-parse','HEAD'],text=True).strip()
assert actual==USALIGN_SHA,f'Unexpected US-align source commit: {actual}'
USALIGN=SRC/'USalign'
if not USALIGN.is_file():subprocess.run(['make'],cwd=SRC,check=True)
assert USALIGN.is_file() and os.access(USALIGN,os.X_OK)
print('✅ US-align pinned:',actual,'gemmi:',gemmi.__version__)

## 3 — Sequence mapping, finite-geometry checks, and scoring parser

In [ ]:
def digest(data):return hashlib.sha256(data).hexdigest()

def seq_from_fasta(data):
    lines=[s.strip() for s in data.decode().splitlines() if s.strip()]
    assert lines and lines[0].startswith('>')
    seq=''.join(s for s in lines if not s.startswith('>')).upper()
    assert seq and set(seq)<=set('ACGU'),f'Unexpected sequence letters: {seq}'
    return seq

def pdb_residues(data):
    seen=set(); result=[]
    for line in data.decode(errors='replace').splitlines():
        if not line.startswith('ATOM  '):continue
        key=(line[21:22],line[22:26],line[26:27],line[17:20].strip())
        if key not in seen:
            seen.add(key)
            result.append({'key':key,'nt':line[17:20].strip(),'position':int(line[22:26])})
    return result

def cif_to_pdb_checked(cif,seq):
    st=gemmi.read_structure(str(cif))
    assert len(st)==1 and len(st[0])==1,f'{cif}: expected one model/one RNA chain'
    chain=st[0][0]; residues=list(chain)
    assert len(residues)==len(seq),f'{cif}: incomplete RNA chain'
    assert ''.join(r.name for r in residues)==seq,f'{cif}: nucleotide sequence mismatch'
    assert [r.seqid.num for r in residues]==list(range(1,len(seq)+1)),f'{cif}: unexpected residue numbering'
    atoms=[a for r in residues for a in r]
    assert atoms and all(math.isfinite(c) for a in atoms for c in (a.pos.x,a.pos.y,a.pos.z)),f'{cif}: NaN/Inf coordinate found'
    pdb=st.make_pdb_string().encode()
    parsed=pdb_residues(pdb)
    assert len(parsed)==len(seq) and ''.join(r['nt'] for r in parsed)==seq
    return pdb,len(atoms)

def map_to_native_positions(pred_pdb,positions):
    records=pdb_residues(pred_pdb)
    index={r['key']:i+1 for i,r in enumerate(records)}
    out=[];serial=1
    for l in pred_pdb.decode(errors='replace').splitlines():
        if not l.startswith(('ATOM  ','HETATM')):continue
        if l[16:17] not in (' ','A','1'):continue
        key=(l[21:22],l[22:26],l[26:27],l[17:20].strip())
        pos=index.get(key)
        if pos not in positions:continue
        out.append(l[:6]+f'{serial:5d}'+l[11:22]+f'{pos:4d}'+l[26:])
        serial+=1
    assert out,'No native-overlapping atomic coordinates'
    return ('\n'.join(out+['TER','END'])+'\n').encode()

def parse_usalign(output,n_native):
    m=re.search(r'Aligned length\s*=\s*(\d+)\s*,\s*RMSD\s*=\s*([\d.eE+-]+)',output,re.I)
    if not m:raise ValueError('US-align missing alignment and RMSD')
    aligned,rmsd=int(m.group(1)),float(m.group(2))
    matches=re.findall(r'TM-score\s*=\s*([\d.eE+-]+)\s*\(normalized by length of Structure_2\b[^)]*\)',output,re.I)
    if not matches:raise ValueError('US-align missing native-normalized (Structure_2) TM-score')
    tm=float(matches[-1])
    assert 0<aligned<=n_native and math.isfinite(rmsd) and rmsd>=0 and math.isfinite(tm) and 0<=tm<=1.00001
    return {'tm_score':tm,'rmsd_angstrom':rmsd,'alignment_residues':aligned,'alignment_coverage':aligned/n_native}

assert parse_usalign('Aligned length= 13, RMSD= 11.52, Seq_ID=n_identical/n_aligned= 1.000\nTM-score= 0.24331 (normalized by length of Structure_2: L=13, d0=0.40)',13)['tm_score']==0.24331
print('✅ RNA mapping and US-align parser self-test passed')

## 4 — Evaluate all ten Protenix structures (no GPU)

In [ ]:
LOCAL=Path('/content')/RUN_NAME
assert not LOCAL.exists(),f'Local evaluation already exists: {LOCAL}'
assert not (EVAL_ROOT/RUN_NAME).exists(),f'Evaluation Drive run already exists: {EVAL_ROOT/RUN_NAME}'
LOCAL.mkdir();(LOCAL/'logs').mkdir();(LOCAL/'mapped').mkdir()
rows=[]
with zipfile.ZipFile(NATIVE_ZIP) as archive:
    for t in TARGETS:
        folder=PRED_ROOT/t
        meta=json.loads((folder/'complete.json').read_text())
        assert meta.get('status')=='PASS' and meta.get('model')=='Protenix' and meta.get('target_id')==t
        assert meta.get('model_name')==MODEL and meta.get('upstream_commit')==UPSTREAM_SHA
        assert meta.get('layernorm_type')=='torch' and meta.get('dtype')=='fp32'
        assert meta.get('trimul_kernel')=='torch' and meta.get('triatt_kernel')=='torch'
        assert meta.get('cache') is False and meta.get('fusion') is False and meta.get('tf32') is False
        assert meta.get('native_used_for_inference') is False and meta.get('n_samples')==1 and meta.get('seed')==101
        artifacts={a['filename']:a for a in meta['artifacts']}
        for filename in ('prediction.cif','input.fasta','summary_confidence.json'):
            assert digest((folder/filename).read_bytes())==artifacts[filename]['sha256'],f'{t}: SHA mismatch {filename}'
        seq_bytes=(folder/'input.fasta').read_bytes()
        assert digest(seq_bytes)==meta['fasta_sha256']
        seq=seq_from_fasta(seq_bytes)
        assert len(seq)==meta['sequence_length']==meta['cif_residues']
        cif=folder/'prediction.cif';pred_pdb,atom_count=cif_to_pdb_checked(cif,seq)
        assert atom_count==meta['atoms_total']==meta['atoms_finite'],f'{t}: invalid atom counts'
        summary=json.loads((folder/'summary_confidence.json').read_text(),parse_constant=lambda x:float('nan'))
        assert all(math.isfinite(float(summary[k])) for k in ('plddt','ptm','ranking_score')),f'{t}: nonfinite confidence values'
        nat_bytes=archive.read(f'native/{t}_native_eval.pdb')
        native_res=pdb_residues(nat_bytes);positions=[r['position'] for r in native_res]
        assert len(positions)==len(set(positions))>0
        assert all(1<=r['position']<=len(seq) and seq[r['position']-1]==r['nt'] for r in native_res),f'{t}: native sequence mapping mismatch'
        mapped=map_to_native_positions(pred_pdb,set(positions))
        mp=LOCAL/'mapped'/f'{t}_protenix_mapped.pdb';np=LOCAL/'mapped'/f'{t}_native_eval.pdb'
        mp.write_bytes(mapped);np.write_bytes(nat_bytes)
        p=subprocess.run([str(USALIGN),str(mp),str(np),'-mol','RNA','-TMscore','1'],capture_output=True,text=True)
        (LOCAL/'logs'/f'{t}_usalign.txt').write_text(p.stdout+'\n--- STDERR ---\n'+p.stderr)
        assert p.returncode==0,f'{t}: US-align failed {p.returncode} {p.stderr[-2000:]}'
        score=parse_usalign(p.stdout,len(native_res))
        row=dict(target_id=t,model='Protenix',model_variant=MODEL,input_condition='RNA_only_no_MSA_no_templates',input_length=len(seq),native_residues=len(native_res),native_fraction=len(native_res)/len(seq),sample_id=0,seed=101,layernorm_type='torch',prediction_sha256=digest(cif.read_bytes()),mapped_sha256=digest(mapped),runtime_seconds=meta['runtime_seconds'],finite_atoms=atom_count,**score,status='scored')
        rows.append(row)
        print(f"{t:9s} TM={score['tm_score']:.5f} RMSD={score['rmsd_angstrom']:.2f} Å aligned={score['alignment_residues']}/{len(native_res)}",flush=True)
assert len(rows)==10
with (LOCAL/'protenix_scores.csv').open('w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(rows[0]),lineterminator='\n');w.writeheader();w.writerows(rows)
summary={'status':'complete','target_count':10,'model':'Protenix','model_variant':MODEL,'model_commit':UPSTREAM_SHA,'configuration':'RNA only; no MSA/RNA-MSA/templates; seed=101; one sample; fp32; LayerNorm=torch; triangle kernels=torch; cache/fusion/tf32 off','usalign_version':'20260527','usalign_source_commit':USALIGN_SHA,'method':'USalign mapped_prediction native_eval -mol RNA -TMscore 1 (Structure_2 native normalization)','mean_tm':statistics.mean(r['tm_score'] for r in rows),'mean_rmsd_angstrom':statistics.mean(r['rmsd_angstrom'] for r in rows),'limitations':['exploratory n=10','varying input conditions versus other models','some experimental RNA chains have incomplete coordinates','training overlap and template scarcity unverified']}
(LOCAL/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
filehashes={str(p.relative_to(LOCAL)):digest(p.read_bytes()) for p in sorted(LOCAL.rglob('*')) if p.is_file()}
(LOCAL/'FINALIZED.json').write_text(json.dumps({'status':'complete','targets':10,'created_utc':datetime.now(timezone.utc).isoformat(),'files':filehashes},indent=2)+'\n')
assert all(digest((LOCAL/k).read_bytes())==v for k,v in filehashes.items())
print(f"✅ LOCAL EVALUATION PASS: 10/10 | mean TM={summary['mean_tm']:.6f} | mean RMSD={summary['mean_rmsd_angstrom']:.3f} Å")

## 5 — Save evaluation and verification hashes to Google Drive

In [ ]:
from zipfile import ZipFile,ZIP_DEFLATED
EVAL_ROOT.mkdir(parents=True,exist_ok=True)
DEST=EVAL_ROOT/RUN_NAME
BACKUP=Path('/content/RNA3D_Protenix_Pilot10_Evaluation_Results.zip')
REMOTE_BACKUP=EVAL_ROOT/BACKUP.name
assert not DEST.exists() and not BACKUP.exists() and not REMOTE_BACKUP.exists(),'Refusing to overwrite previous results'
with ZipFile(BACKUP,'x',compression=ZIP_DEFLATED,compresslevel=6) as z:
    for p in sorted(LOCAL.rglob('*')):
        if p.is_file():z.write(p,arcname=RUN_NAME+'/'+str(p.relative_to(LOCAL)))
with ZipFile(BACKUP) as z:assert z.testzip() is None
shutil.copytree(LOCAL,DEST)
shutil.copy2(BACKUP,REMOTE_BACKUP)
for p in LOCAL.rglob('*'):
    if p.is_file():
        other=DEST/p.relative_to(LOCAL)
        assert other.is_file() and digest(other.read_bytes())==digest(p.read_bytes()),f'Saved Drive mismatch: {p}'
assert digest(BACKUP.read_bytes())==digest(REMOTE_BACKUP.read_bytes())
print('✅ FINAL DRIVE VERIFICATION PASS: 10/10 Protenix scores saved')
print('Drive results:',DEST)
print('Backup ZIP:',REMOTE_BACKUP)

## 6 — Final read-only verification

In [ ]:
SAVED=EVAL_ROOT/RUN_NAME
record=json.loads((SAVED/'FINALIZED.json').read_text())
assert record['status']=='complete' and record['targets']==10
for rel,checksum in record['files'].items():
    p=SAVED/rel
    assert p.is_file() and digest(p.read_bytes())==checksum,f'Missing or changed artifact: {rel}'
with (SAVED/'protenix_scores.csv').open() as f:
    checked=list(csv.DictReader(f))
assert len(checked)==10 and {r['target_id'] for r in checked}==set(TARGETS)
assert all(r['status']=='scored' for r in checked)
print('FINAL DRIVE VERIFICATION PASS: 10/10 PROTENIX SCORES INTACT')
print(json.loads((SAVED/'summary.json').read_text()))